# Blog Search Agent — Mode Testing

Interactive notebook to test both modes of the blog search agent:

1. **Chat mode** — conversational, free-form natural language, multi-turn capable
2. **Workflow mode** — deterministic pipeline, structured JSON output with Pydantic validation

### Prerequisites
- AWS credentials configured (`AWS_PROFILE` or env vars)
- `.env` file with `GATEWAY_URL`, `COGNITO_*` vars (or `GATEWAY_TOKEN`)
- `uv sync` to install dependencies

In [1]:
import sys
import os
import json
import asyncio
import logging
from pathlib import Path

# Ensure blog_search is importable
sys.path.insert(0, str(Path.cwd()))

# Force suppress all INFO/DEBUG logs — only show ERROR
logging.basicConfig(level=logging.ERROR, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", force=True)
logging.getLogger().setLevel(logging.ERROR)

In [2]:
# # One-time setup: Create a memory resource (run once, then add MEMORY_ID to .env)
# from bedrock_agentcore.memory.controlplane import MemoryControlPlaneClient

# memory_client = MemoryControlPlaneClient(region_name="us-east-1")
# memory = memory_client.create_memory(
#     name="blogSearchSessionMemory",
#     description="Short-term session memory for blog search agent",
#     event_expiry_days=30,
#     wait_for_active=True,
# )
# print(f"Memory created! Add this to your .env file:")
# print(f"MEMORY_ID={memory['id']}")

## Configuration

- **Chat mode**: `CHAT_LOOKBACK_HOURS` — how many hours back to search for events. The agent searches for events in a datetime window from (now - chat_lookback_hours) to now.
- **Workflow mode**: `WORKFLOW_LOOKBACK_HOURS` — how many hours back from `WORKFLOW_REFERENCE_DATE` to search
- **Reference date**: Set to a past date (e.g. `"2026-07-15"`) for testing, or `""` for real current date

In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CONFIGURATION — Change these values to control test behavior
# ═══════════════════════════════════════════════════════════════════════

# Chat mode: how many hours back to search
CHAT_LOOKBACK_HOURS = 168

# Workflow mode: how many hours back from reference_date to search
WORKFLOW_LOOKBACK_HOURS = 1.5  # 0 = reference_date only, 1.5 = past 90 min, 24 = past day

# Debug mode — prints full tool traces and system prompts
DEBUG = False

print(f"Chat lookback: {CHAT_LOOKBACK_HOURS} hours")
print(f"Workflow lookback: {WORKFLOW_LOOKBACK_HOURS}")
print(f"Debug: {DEBUG}")

Chat lookback: 168 hours
Workflow lookback: 1.5
Debug: False


In [4]:
from blog_search_agent import (
    create_chat_agent,
    run_blog_search_workflow,
    extract_json_from_response,
)
from models import EVENT_TYPES, validate_response

print(f"Event types: {EVENT_TYPES}")

Event types: ['INJURY', 'CANCELLATION', 'VENUE_CHANGE', 'SCHEDULE_CHANGE', 'ROSTER']


In [5]:
async def stream_response(agent, input_query):
    async for event in agent.stream_async(input_query):
        if "data" in event:
            print(event["data"], end="", flush=True)

---
## 1. Chat Mode

Chat mode is conversational — it accepts free-form questions and responds with cited, grounded answers.
The agent has access to: `registry_lookup`, `rss_fetch`, `web_fetch`, `gateway__WebSearch`.

## Create a chat agent with streaming output
chat_agent = create_chat_agent(lookback_hours=CHAT_LOOKBACK_HOURS, debug=DEBUG)

## Stream the response — tokens appear as they're generated
print("Streaming response:\n")
async for event in chat_agent.stream_async("Find any recent injury news for Colgate Raiders basketball"):
    if "data" in event:
        print(event["data"], end="", flush=True)
print()

In [6]:
# Create a chat agent (no memory — local testing without AgentCore session)
chat_agent = create_chat_agent(lookback_hours=CHAT_LOOKBACK_HOURS, debug=False)
input_query = "Find any recent injury news for Colgate Raiders Men's basketball"
# Ask about recent events for a team
print("Streaming response:\n")
await stream_response(chat_agent, input_query)


Streaming response:

Looking up blog sources for Colgate Raiders...Found 4 sources. Checking RSS feeds for recent posts...No injury events found in RSS feeds. Searching the web for more recent news...I found a relevant Yahoo Sports injury page. Let me check it directly for current injury news.

Reading article at https://sports.yahoo.com/ncaab/teams/colgate/injuries/...---

## 🏀 Colgate Raiders Men's Basketball — Injury Report (July 20–27, 2026)

**No injury events were found** within the current search window (July 20–27, 2026) across all sources checked:

| Source | Result |
|---|---|
| 🔵 Colgate Athletics RSS (`colgateathletics.com`) | No injury events |
| 📰 Colgate Maroon News RSS (`thecolgatemaroonnews.com`) | No entries in window |
| 🗣️ RaidersTalk Fan Blog RSS (`raiderstalk.com`) | Feed unavailable |
| 🌐 Yahoo Sports NCAAB Injuries (`sports.yahoo.com`) | No injury events |

### What Was Found
The most recent notable news for the Raiders is a **roster/recruiting update** from Jun

### 1.2 Follow-up question (same agent instance = same conversation context)

In [7]:
# Follow-up on the same agent — tests multi-turn within a session

await stream_response(chat_agent, "okay, then give the exact snippets and sources for roster change?")

Based on the information already retrieved, here is the exact snippet and source for the Colgate Raiders roster news:

---

### 📋 Roster Move — Colgate Raiders Men's Basketball

**Source:** [https://colgateathletics.com/news/2026/6/15/mens-basketball-mens-basketball-announces-incoming-class.aspx](https://colgateathletics.com/news/2026/6/15/mens-basketball-mens-basketball-announces-incoming-class.aspx)
**Published:** June 16, 2026

**Exact Snippet:**
> *"Men's Basketball Announces Incoming Class — Three Raiders to join the program in 2026-27. HAMILTON – Colgate University Head Men's Basketball Coach has announced three additions to the Raiders' roster for the upcoming 2026-27 season.*
>
> **Jayden Harrington** G | 6-4 | 180 | Glen Ridge, Mass. / New Hampton School — A four-year varsity letterwinner...transferred to New Hampton after competing for Seton Hall Prep...appointed team captain for the 2025-26 senior year...two-time NEPSAC Honorable Mention...recipient of the Coaches Award.
>
>

### 1.3 Out-of-scope request

In [8]:
# Out-of-scope: should respond politely and explain what it can help with
await stream_response(chat_agent, "What's the weather like in New York today?")


I specialize in **NCAA Men's Basketball event extraction**. I'm not able to help with weather queries, but I can assist you with finding the following types of information for NCAA Men's Basketball teams:

- 🏥 **Injury news**
- 📅 **Schedule changes**
- 🏟️ **Venue updates**
- 📋 **Roster moves**
- ❌ **Game cancellations**

Would you like me to search for any of the above for a specific team?

### 1.4 Greeting (conversational handling)

In [9]:
# Greeting: should respond conversationally, not attempt tool calls
await stream_response(chat_agent, "Hi! How are you doing today?")

I'm doing great, thanks for asking! 😊 However, I'm specialized in **NCAA Men's Basketball event extraction**, so I'm best put to use helping you track down things like:

- 🏥 **Injury news**
- 📅 **Schedule changes**
- 🏟️ **Venue updates**
- 📋 **Roster moves**
- ❌ **Game cancellations**

Just give me a team name and I'll get searching! Is there a team you'd like me to look into?

### 1.5 Specific event type request

In [10]:
# Ask for a specific event type in user-provided URLs
specific_agent = create_chat_agent(lookback_hours=CHAT_LOOKBACK_HOURS, debug=DEBUG)
query="""Are there any roaster news or injury news for Duke Blue Devils 
in https://balldurham.com/ or 
https://forums.dukebasketballreport.com/index.php?forums/elizabeth-king-forum.2/ in the past 2 months?"""
await stream_response(chat_agent, query)


Since the user provided specific URLs, I'll fetch those directly. Computing the date range: past 2 months from now = **2026-05-27 to 2026-07-27**.

Reading article at https://balldurham.com/...
Reading article at https://forums.dukebasketballreport.com/index.php?forums/elizabeth-king-forum.2/...Here are the results for **Duke Blue Devils Men's Basketball** across both sources (May 27 – July 27, 2026):

---

### 🏀 Duke Blue Devils — Injury & Roster News (Past 2 Months)

---

#### 📋 ROSTER — Event 1
- **Player:** Joaquim Boumtje Boumtje
- **Date:** April 30, 2026 ⚠️ *(just outside the 2-month window)*
- **Excerpt:** *"Welcome to Duke, Joaquim Boumtje Boumtje!"*
- **Summary:** Joaquim Boumtje Boumtje joined the Duke Blue Devils roster.
- **Source:** [Duke Basketball Report Forum](https://forums.dukebasketballreport.com/index.php?forums/elizabeth-king-forum.2/)

---

#### 📋 ROSTER — Event 2
- **Player:** N/A (team-wide)
- **Date:** May 28, 2026 ✅ *(within window)*
- **Excerpt:** *"MBB: 202

In [23]:
query="""Are there any roaster news or announcements for Wildcats in https://acusports.com/rss?path=mbball between July 10 - July 20, 2026?"""
await stream_response(chat_agent, query)

The user is asking about roster news and announcements. Since "announcements" is not one of the defined event types I cover (INJURY, CANCELLATION, VENUE_CHANGE, SCHEDULE_CHANGE, ROSTER), I'll search for the relevant supported types. Let me re-fetch the RSS feed with all applicable event types.

Checking RSS feed at https://acusports.com/rss?path=mbball...---

### 🏀 ACU Wildcats Men's Basketball — Roster News (July 10–20, 2026)

---

#### 📋 ROSTER — Coaching Staff Announcement
- **Date:** July 17, 2026 ✅
- **Excerpt:** *"Abilene Christian men's basketball head coach Brete Tanner has announced the Wildcats' coaching staff for the 2026-27 season."*
- **Summary:** Head coach Brete Tanner announced the full coaching staff for the upcoming 2026-27 season.
- **Source:** [https://acusports.com/rss?path=mbball](https://acusports.com/rss?path=mbball)

---

| Detail | Value |
|---|---|
| **Total entries in feed** | 10 |
| **Entries within July 10–20 window** | 4 |
| **Roster/Announcement events f

# AgentCore Memory Test

In [19]:
# Session-scoped memory test — requires MEMORY_ID in .env
# Uses a fixed session_id so you can re-run and see history restored

SESSION_ID = "test-session-005"
USER_ID = "test-user-local"

# Turn 1: Ask a question
memory_agent = create_chat_agent(
    user_id=USER_ID,
    session_id=SESSION_ID,
    lookback_hours=CHAT_LOOKBACK_HOURS,
    debug=DEBUG,
)
question = "Find any injury news for Duke Blue Devils"
print(f"turn 1 question: {question}\nResponse: ")
await stream_response(memory_agent, question)


# Turn 2: Follow-up (same session — history should be available)
followup_question = "what about Vermont Catamounts?"
print(f"\nTURN 2 (follow-up): {followup_question}\nResponse: ")
await stream_response(memory_agent, followup_question)

turn 1 question: Find any injury news for Duke Blue Devils
Response: 
Looking up blog sources for Duke Blue Devils...Found 2 RSS feeds and several blog sources. Checking RSS feeds and searching the web simultaneously...The RSS feeds had no injury events in the target window, but the web search returned some relevant snippets. Let me now check the most promising articles within the search window and also fetch the balldurham.com blog for any recent injury news.---

## Duke Blue Devils — Injury News Summary

**Search Window: July 20–27, 2026**

After checking all available sources — RSS feeds (Duke Basketball Report, Pine State Playbook), the Ball Durham blog, and the SI.com article from July 26 — **no new injury events were found within the July 20–27, 2026 window.**

---

### 📋 Background Context (Outside the Search Window)

While no current injuries were flagged, the web search did surface notable injury history from earlier in 2026 that may be relevant for context:

| Player | Injury

In [20]:
# Cross-instance test: NEW agent instance, SAME session_id
# If memory is working, this agent should "remember" the previous conversation
memory_agent_2 = create_chat_agent(
    user_id=USER_ID,
    session_id=SESSION_ID,  # same session — should restore history
    lookback_hours=CHAT_LOOKBACK_HOURS,
    debug=DEBUG,
)
followup_question_2 = "What did we discuss earlier about Duke?"
print(f"\nTURN 3 (follow-up with new agent instance, same session_id): {followup_question}\nResponse: ")
await stream_response(memory_agent_2, followup_question_2)


TURN 3 (follow-up with new agent instance, same session_id): what about Vermont Catamounts?
Response: 
Based on our earlier conversation, here's a summary of what we found for the **Duke Blue Devils**:

---

## Duke Blue Devils — Injury News Recap

**Search Window: July 20–27, 2026**

**No new injury events were found within the July 20–27, 2026 search window** across all sources checked, including:
- RSS feed: Duke Basketball Report Forum
- RSS feed: Pine State Playbook (Substack)
- Ball Durham blog (SB Nation)
- SI.com (July 26, 2026 article)

---

### 📋 Background Injury Context (Earlier in 2026)

Two notable injuries were surfaced from earlier in the season:

| Player | Injury | Timeline |
|---|---|---|
| **Caleb Foster** (#1, G) | Right foot fracture — underwent surgery March 8, 2026 | Out several weeks; listed as questionable for Elite Eight (March 29). Appears to have recovered — back as a senior for 2026-27. |
| **Patrick Ngongba II** (#21, C) | Undisclosed leg injury (walking

---
## 2. Workflow Mode

Workflow mode is a deterministic pipeline for batch/scheduled runs:

 Step 1: Registry lookup (plain Python, no LLM)
    Fetches team's blog URLs from DynamoDB

    Only returns crawlable + non-outdated URLs

Step 2: RSS feeds — parallel, no agent loop

    All RSS URLs are called concurrently via asyncio.gather

    Each rss_fetch call: fetches the feed → Nova 2 Lite extracts events → returns structured events

    Events are validated against EventResult Pydantic schema; malformed ones are dropped

Step 3: Non-RSS URLs — agent loop (sequential reasoning, parallel tool execution)

    A Sonnet agent is created with gateway__WebSearch + web_fetch tools
    
    The agent decides search queries and which URLs to fetch (requires LLM reasoning)

    When the agent requests multiple web_fetch calls in one turn, Strands executes them concurrently (via ConcurrentToolExecutor)

    But each agent "turn" (think → call tools → think) is sequential

Step 4: Aggregate all events and return

So: RSS is fully parallel (all feeds at once). Non-RSS is agent-driven — parallel within a single turn, but sequential across turns.

### 2.1 Standard workflow run (today only — default)

In [13]:
result = await run_blog_search_workflow(
    team="Duke Blue Devils",
    lookback_hours=WORKFLOW_LOOKBACK_HOURS,
    reference_date="2026-03-10",
    debug=False,
)

print("\n" + "="*70)
print("WORKFLOW RESULT:")
print("="*70)
print(json.dumps(result, indent=2, default=str))


WORKFLOW RESULT:
{
  "status": "success",
  "mode": "workflow",
  "team": "Duke Blue Devils",
  "sport": "CBB",
  "lookback_hours": 1.5,
  "date_range": {
    "start": "2026-03-09T22:30:00+00:00",
    "end": "2026-03-10T00:00:00+00:00"
  },
  "results": [
    {
      "sport": "CBB",
      "team": "Duke Blue Devils",
      "event_type": "INJURY",
      "player_name": "Patrick Ngongba II",
      "excerpt": "Duke center Patrick Ngongba II is likely out for the conference tournament, so now, the Blue Devils are turning their attention to this big man. As the Blue Devils took down the North Carolina Tar Heels, Duke starting center Patrick Ngongba II was on the bench and wearing a walking boot on his right foot.",
      "summary": "Duke center Patrick Ngongba II suffered an injury to his right foot and is likely out for the ACC tournament.",
      "source_url": "https://balldurham.com/patrick-ngongba-injury-all-eyes-on-this-duke-big-man-with-acc-tournament-next",
      "blog_post_date": nul

### 2.2 Validate the output schema

In [14]:
if result.get("status") == "success":
    # Re-validate the returned data against Pydantic models
    validation = validate_response({
        "results": result["results"],
        "retrieval_diagnostics": result["retrieval_diagnostics"],
    })
    if isinstance(validation, list):
        print("VALIDATION ERRORS:")
        for err in validation:
            print(f"  - {err}")
    else:
        print(f"Validation PASSED: {len(validation.results)} events")
        for ev in validation.results:
            print(f"  [{ev.event_type}] {ev.summary[:80]}")
else:
    print(f"Workflow returned error: {result.get('error')}")

Validation PASSED: 6 events
  [INJURY] Duke center Patrick Ngongba II suffered an injury to his right foot and is likel
  [INJURY] Caleb Foster suffered a right foot fracture and underwent surgery, ruling him ou
  [INJURY] Caleb Foster suffered a right foot fracture requiring surgery and is out for the
  [INJURY] Pat Ngongba II is out for the ACC Tournament due to foot soreness with hopes of 
  [INJURY] Duke starting point guard Caleb Foster suffered a broken right foot requiring su
  [INJURY] Duke starting big man Patrick Ngongba II will sit out the ACC Tournament due to 


### 2.5 Workflow with non-existent team (error path)

In [15]:
result_err = await run_blog_search_workflow(
    team="Nonexistent University Falcons",
    sport="NCAA Men's Basketball",
    lookback_hours=WORKFLOW_LOOKBACK_HOURS,
    reference_date="",
    debug=DEBUG,
)

print(f"Status: {result_err.get('status')}")
print(f"Error: {result_err.get('error', 'N/A')}")

Status: error
Error: Team 'Nonexistent University Falcons' not found in registry


---
## 4. JSON Extraction (Unit Tests)

Test the balanced-brace JSON extraction from agent response text.

In [16]:
# JSON embedded in markdown code block
text_with_codeblock = '''Here are the results:
```json
{"results": [], "retrieval_diagnostics": {"urls_total": 0, "urls_with_rss": 0, "urls_without_rss": 0, "web_searches_performed": 0, "rss_feeds_fetched": 0, "web_fetches_performed": 0}}
```
'''
parsed = extract_json_from_response(text_with_codeblock)
assert parsed is not None
assert parsed["results"] == []
print("Code block extraction: PASSED")

# Raw JSON (no wrapping)
raw_json = '{"results": [{"sport": "Basketball"}], "retrieval_diagnostics": {"urls_total": 1, "urls_with_rss": 0, "urls_without_rss": 1, "web_searches_performed": 0, "rss_feeds_fetched": 0, "web_fetches_performed": 0}}'
parsed = extract_json_from_response(raw_json)
assert parsed is not None
assert len(parsed["results"]) == 1
print("Raw JSON extraction: PASSED")

# JSON with preamble text
text_with_preamble = 'Based on my search, here is the result: {"results": [], "retrieval_diagnostics": {"urls_total": 2, "urls_with_rss": 1, "urls_without_rss": 1, "web_searches_performed": 1, "rss_feeds_fetched": 1, "web_fetches_performed": 0}}'
parsed = extract_json_from_response(text_with_preamble)
assert parsed is not None
assert parsed["retrieval_diagnostics"]["urls_total"] == 2
print("Preamble extraction: PASSED")

# Non-JSON response (conversational)
conversational = "I couldn't find any recent injury news for that team. Would you like me to check another team?"
parsed = extract_json_from_response(conversational)
assert parsed is None
print("Non-JSON response: correctly returned None")

Code block extraction: PASSED
Raw JSON extraction: PASSED
Preamble extraction: PASSED
Non-JSON response: correctly returned None


---
## 5. RSS Datetime Filtering Tests

Verify that `rss_fetch` correctly filters feed entries by datetime window,
and that the chat agent passes the correct 24-hour window to tools.

### 5.1 Unit test: rss_fetch filters entries by datetime window

Call `rss_fetch` directly with a tight 1.5-hour window. Verify that:
- `entries_total` reflects all entries in the feed
- `entries_in_window` <= `entries_total` (some entries filtered out)
- No LLM call is made if `entries_in_window == 0`

In [17]:
from datetime import datetime, timedelta, timezone
from tools.rss_fetch import rss_fetch, _parse_datetime, _is_within_window

# Pick an active RSS feed from registry
TEST_RSS_URL = "https://acusports.com/rss?path=mbball"  # Sidearm Sports feed

now = datetime.now(timezone.utc)

# Test 1: Tight window (1.5 hours) — most entries should be filtered out
dt_start_tight = (now - timedelta(hours=1.5)).isoformat(timespec="seconds")
dt_end = now.isoformat(timespec="seconds")

print(f"Test 1: Tight window (last 1.5 hours)")
print(f"  datetime_start: {dt_start_tight}")
print(f"  datetime_end:   {dt_end}")

result_tight = await rss_fetch(
    rss_url=TEST_RSS_URL,
    team="Abilene Christian Wildcats",
    datetime_start=dt_start_tight,
    datetime_end=dt_end,
    event_types="INJURY,ROSTER",
)

print(f"  entries_total:     {result_tight['entries_total']}")
print(f"  entries_in_window: {result_tight['entries_in_window']}")
print(f"  N events extracted:  {len(result_tight['events'])}")
print(f"  Events:  {result_tight['events']}")
print(f"  error:             {result_tight.get('error')}")
assert result_tight["entries_in_window"] <= result_tight["entries_total"]

print("  \u2713 PASSED: entries_in_window \u2264 entries_total")

# Test 2: Wide window (30 days) — all/most entries should pass
dt_start_wide = (now - timedelta(days=30)).isoformat(timespec="seconds")
dt_end = now.isoformat(timespec="seconds")

print(f"\nTest 2: Wide window (last 30 days)")
print(f"  datetime_start: {dt_start_wide}")
print(f"  datetime_end:   {dt_end}")

result_wide = await rss_fetch(
    rss_url=TEST_RSS_URL,
    team="Abilene Christian Wildcats",
    datetime_start=dt_start_wide,
    datetime_end=dt_end,
    event_types="INJURY,ROSTER",
)

print(f"  entries_total:     {result_wide['entries_total']}")
print(f"  entries_in_window: {result_wide['entries_in_window']}")
print(f"  events extracted:  {len(result_wide['events'])}")
print(f"  Events:  {result_wide['events']}")
assert result_wide["entries_in_window"] >= result_tight["entries_in_window"]
print("  \u2713 PASSED: wider window includes more entries")

# Test 3: Window in the distant past — should get 0 entries in window
dt_start_old = "2020-01-01T00:00:00+00:00"
dt_end_old = "2020-01-02T00:00:00+00:00"

print(f"\nTest 3: Historical window (2020-01-01) — expect 0 entries")

result_old = await rss_fetch(
    rss_url=TEST_RSS_URL,
    team="Abilene Christian Wildcats",
    datetime_start=dt_start_old,
    datetime_end=dt_end_old,
    event_types="INJURY,ROSTER",
)

print(f"  entries_total:     {result_old['entries_total']}")
print(f"  entries_in_window: {result_old['entries_in_window']}")
print(f"  events extracted:  {len(result_old['events'])}")
assert result_old["entries_in_window"] == 0
assert result_old["events"] == []
print("  \u2713 PASSED: no entries in historical window, LLM call skipped")

Test 1: Tight window (last 1.5 hours)
  datetime_start: 2026-07-27T14:56:12+00:00
  datetime_end:   2026-07-27T16:26:12+00:00
  entries_total:     10
  entries_in_window: 0
  N events extracted:  0
  Events:  []
  error:             None
  ✓ PASSED: entries_in_window ≤ entries_total

Test 2: Wide window (last 30 days)
  datetime_start: 2026-06-27T16:26:12+00:00
  datetime_end:   2026-07-27T16:26:12+00:00
  entries_total:     10
  entries_in_window: 6
  events extracted:  2
  Events:  [{'sport': "NCAA Men's Basketball", 'team': 'Abilene Christian Wildcats', 'event_type': 'ROSTER', 'player_name': None, 'excerpt': "Abilene Christian men's basketball head coach Brete Tanner has announced the Wildcats' coaching staff for the 2026-27 season.", 'summary': 'Abilene Christian announced their coaching staff for the 2026-27 season.', 'source_url': 'https://acusports.com/rss?path=mbball', 'blog_post_date': '2026-07-17', 'detected_at': '2026-07-27T16:26:12+00:00', 'retrieval_method': 'rss'}, {'spor